In [2]:
import warnings

from tqdm import tqdm

warnings.filterwarnings("ignore")
import os
import shutil

import matplotlib
import pandas as pd

import codecs  # this is used for file operations
import gc
import multiprocessing
import pickle
import pickle as pkl
import random as r
import re
from datetime import datetime as dt
from multiprocessing import Process  # this is used for multithreading

import dask.dataframe as dd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import scipy.sparse
import seaborn as sns
from itertools import product
from pathlib import Path
from nltk.util import ngrams
from sklearn import preprocessing
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, chi2, f_regression
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, log_loss
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
# Function to return all possible n*n*n combinations of trigrams
def calculate_trigram(tokens):
    sent = ""
    trigram_result = []
    for i in range(len(tokens)):
        for j in range(len(tokens)):
            for k in range(len(tokens)):
                trigram = tokens[i] + " " + tokens[j] + " " + tokens[k]
                trigram_result.append(trigram)
    return trigram_result
  

# test_tokens=['edx','esi','eax']
# trigram_result = calculate_trigram(test_tokens)
# trigram_result

In [4]:
opcodes_trigram = ['jmp', 'mov', 'retf', 'push', 'pop', 'xor', 'retn', 'nop', 'sub', 'inc', 'dec', 'add','imul', 'xchg', 'or', 'shr', 'cmp', 'call', 'shl', 'ror', 'rol', 'jnb','jz','rtn','lea','movzx']

opcodes_trigram_asm_vocabulary = calculate_trigram(
    opcodes_trigram
)  # Holding all n*n*n possible combinations of trigrams_from_asm_files

vectorizer = CountVectorizer(
    tokenizer=lambda x: x.split(),
    lowercase=False,
    ngram_range=(3, 3),
    vocabulary=opcodes_trigram_asm_vocabulary,
)  # NOTE: without "tokenizer=lambda x: x.split()", "??" would not get vectorized properly

file_lists_asm_opcodes = os.listdir("opcodes_asm_files")
feature_names_array = vectorizer.get_feature_names_out()
features = ["ID"] + list(feature_names_array)


opcodes_asm_trigram_df = pd.DataFrame(columns=features)

with open(
    "featurization/opcodes_asm_trigram_df.csv", mode="w"
) as opcodes_asm_trigram_df:
    
    opcodes_asm_trigram_df.write(",".join(map(str, features)))
    
    opcodes_asm_trigram_df.write("\n")
    
    for _, current_asm_textized_file in tqdm(enumerate(file_lists_asm_opcodes)):
        each_file_id = current_asm_textized_file.split("_")[0]
        current_asm_textized_file = open(
            "opcodes_asm_files/" + current_asm_textized_file
        )
        corpus_for_asm_files_opcodes = [
            current_asm_textized_file.read().replace("\n", " ").lower()
        ]  # This will contain all the opcodes_trigram codes for a given current_asm_textized_file

        # CountVectorizer produces a sparse representation of the counts using scipy.sparse.csr_matrix.
        # Hence below is a sparse vector of all trigram counts from corpus_for_asm_files_opcodes
        trigrams_from_asm_files = vectorizer.transform(corpus_for_asm_files_opcodes)

        # So now return a dense ndarray representation of this matrix
        # Updating each row_trigram_count of the dataframe with trigram counts
        # of corresponding current_asm_textized_file
        row_trigram_count = scipy.sparse.csr_matrix(trigrams_from_asm_files).toarray()

        # Write that single row in the CSV for current_asm_textized_file
        opcodes_asm_trigram_df.write(
            ",".join(map(str, [each_file_id] + list(row_trigram_count[0])))
        )

        opcodes_asm_trigram_df.write("\n")

        current_asm_textized_file.close()

10868it [01:35, 114.21it/s]


,ID,jmp jmp jmp,jmp jmp mov,jmp jmp retf,jmp jmp push,jmp jmp pop,jmp jmp xor,jmp jmp retn,jmp jmp nop,jmp jmp sub,...,movzx movzx cmp,movzx movzx call,movzx movzx shl,movzx movzx ror,movzx movzx rol,movzx movzx jnb,movzx movzx jz,movzx movzx rtn,movzx movzx lea,movzx movzx movzx
0,01azqd4InC7m9JpocGv5,437,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,01IsoiSMh5gxyDYTl4CB,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,01jsnpXSAlgw6aPeDxrU,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,1,0,1,1
3,01kcPWA9K2BOxQeS5Rju,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,01SuzwMJEIXsK7A8dQbl,2,1,1,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [3]:
opcodes_asm_trigram_df = pd.read_csv(
    "featurization/opcodes_asm_trigram_df.csv"
)
opcodes_asm_trigram_df.head()

,ID,jmp jmp jmp,jmp jmp mov,jmp jmp retf,jmp jmp push,jmp jmp pop,jmp jmp xor,jmp jmp retn,jmp jmp nop,jmp jmp sub,...,movzx movzx cmp,movzx movzx call,movzx movzx shl,movzx movzx ror,movzx movzx rol,movzx movzx jnb,movzx movzx jz,movzx movzx rtn,movzx movzx lea,movzx movzx movzx
0,01azqd4InC7m9JpocGv5,437,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,01IsoiSMh5gxyDYTl4CB,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,01jsnpXSAlgw6aPeDxrU,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,1,0,1,1
3,01kcPWA9K2BOxQeS5Rju,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,01SuzwMJEIXsK7A8dQbl,2,1,1,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [4]:
# 1. 讀取標籤並合併 (這時候 final_df 包含所有資料)
with open('featurization/class_labels.pkl', 'rb') as file:
    class_labels = pkl.load(file)
y_labels = class_labels.rename(columns={"Id": "ID", "Class": "Class"})
full_merged_df = pd.merge(opcodes_asm_trigram_df, y_labels, on="ID", how="inner")

# 2. 讀取 ID 清單
train_ids_list = pd.read_csv("featurization/train_ids")["ID"]
test_ids_list = pd.read_csv("featurization/test_ids")["ID"]

# 3. 分別切出訓練集與測試集 (從 full_merged_df 切分)
train_df = full_merged_df[full_merged_df['ID'].isin(train_ids_list)].sort_values('ID')
test_df = full_merged_df[full_merged_df['ID'].isin(test_ids_list)].sort_values('ID')

# 4. 準備特徵 (X) 與標籤 (y)
X_train = train_df.drop(["ID", "Class"], axis=1)
y_train = train_df["Class"]
X_test = test_df.drop(["ID", "Class"], axis=1)
y_test = test_df["Class"]

# 5. 特徵選擇：Fit (只針對訓練集)
select_kbest_object = SelectKBest(score_func=chi2, k=500)
select_kbest_object.fit(X_train, y_train)

# 6. 特徵選擇：Transform (套用到兩邊，這步最關鍵！)
# 這會把特徵數從幾萬個縮減到 2000 個
X_train_reduced = select_kbest_object.transform(X_train)
X_test_reduced = select_kbest_object.transform(X_test)

# 7. (選做) 建立分數表，方便觀察
most_imp_byte_bigram_feature_score_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Score': select_kbest_object.scores_
}).sort_values(by='Score', ascending=False)

# 被選中的column name
selected_mask = select_kbest_object.get_support()
selected_feature_names = X_train.columns[selected_mask]

# 完整的train df 包含ID
X_train_reduced_df = pd.DataFrame(
    X_train_reduced, 
    columns=selected_feature_names, 
    index=X_train.index
)
X_train_reduced_df.insert(0, 'ID', train_df['ID'])

# 完整的test df 包含ID
X_test_reduced_df = pd.DataFrame(
    X_test_reduced, 
    columns=selected_feature_names, 
    index=X_test.index
)
X_test_reduced_df.insert(0, 'ID', test_df['ID'])

X_train_reduced_df.to_csv("featurization/featurization_final/top_800_asm_opcodes_trigram_df_train.csv",index=False)
X_test_reduced_df.to_csv("featurization/featurization_final/top_800_asm_opcodes_trigram_df_test.csv",index=False)

In [7]:
%%time 
with open('featurization/class_labels.pkl', 'rb') as file:
    class_labels=pkl.load(file)

X_opcode_asm_trigram = opcodes_asm_trigram_df
y = class_labels
y = y.rename(columns={"Id": "ID", "Class": "Class"})
final_df = pd.merge(opcodes_asm_trigram_df, y, on="ID", how="inner")

X = final_df.drop(["ID", "Class"], axis=1)
y = final_df["Class"]
  

# X_opcode_asm_trigram.head()

#Get the best 800 features using SelectKBest. Save the feature scores along with the feature names in a feature_score_df_df, which we will use to get the best fetures from the bigrams df data

kbest_object = SelectKBest(score_func=chi2, k=800)

top_features=kbest_object.fit(X_opcode_asm_trigram.drop("ID", axis=1), y)

top_features_scores=pd.DataFrame(top_features.scores_)

X_opcode_columns=pd.DataFrame(X_opcode_asm_trigram.columns)

top_asm_opcode_trigram_df=pd.concat([X_opcode_columns,top_features_scores],axis=1)

top_asm_opcode_trigram_df.columns=["ASM_Opcode_Top_Feature_Name","ASM_Opcode_Top_Feature_Score"]

top_asm_opcode_trigram_df=top_asm_opcode_trigram_df.nlargest(800,"ASM_Opcode_Top_Feature_Score")

top_asm_opcode_trigram_df.head()

CPU times: total: 6.44 s
Wall time: 9.22 s


,ASM_Opcode_Top_Feature_Name,ASM_Opcode_Top_Feature_Score
703,mov mov jmp,1.374010e+07
2109,push push retf,6.004949e+06
714,mov mov add,4.967466e+06
8139,imul mov jmp,4.928314e+06
989,mov imul jmp,4.812556e+06


In [8]:
%%time

# Get List of the 800 top features
top_800_asm_trigram_features=list(top_asm_opcode_trigram_df["ASM_Opcode_Top_Feature_Name"])

top_800_asm_trigam_df=pd.concat([X_opcode_asm_trigram["ID"], X_opcode_asm_trigram[top_800_asm_trigram_features]], axis=1)

# The "ID" column was being duplicated, hence need to remove that, and also the possibility of any other duplicated column
top_800_asm_trigam_df = top_800_asm_trigam_df.loc[:,~top_800_asm_trigam_df.columns.duplicated()]

top_800_asm_trigam_df.to_csv("featurization/featurization_final/top_800_asm_opcodes_trigram_df.csv",index=False)

top_800_asm_trigam_df.head()

CPU times: total: 1.22 s
Wall time: 1.29 s


,ID,mov mov jmp,push push retf,mov mov add,imul mov jmp,mov imul jmp,push push cmp,push call jmp,mov push retf,push mov retf,...,cmp add jmp,cmp lea cmp,sub add jmp,jmp add jmp,cmp lea shr,add mov or,add mov sub,mov movzx dec,retf retf jmp,jz rtn movzx
0,01azqd4InC7m9JpocGv5,37,0,131,6,1,1,16,0,0,...,2,0,0,4,0,3,5,0,0,0
1,01IsoiSMh5gxyDYTl4CB,4,0,78,0,0,0,1,0,0,...,0,0,0,0,0,2,10,1,0,0
2,01jsnpXSAlgw6aPeDxrU,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,01kcPWA9K2BOxQeS5Rju,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,01SuzwMJEIXsK7A8dQbl,20,0,305,0,0,0,0,0,0,...,0,0,0,0,0,0,4,0,0,0
